In [ ]:
from pathlib import Path
import re
import pandas as pd

# =========================
# Parameters to edit
# =========================

def find_solution_dirs():
    current = Path.cwd().resolve()

    # Walk up the directory tree
    for parent in [current] + list(current.parents):
        vanilla = list(parent.rglob("vanilla_subset*"))
        sfc = list(parent.rglob("sfc_subset*"))

        # Keep only directories
        vanilla = [p for p in vanilla if p.is_dir()]
        sfc = [p for p in sfc if p.is_dir()]

        if vanilla and sfc:
            print("Found solution folders:")
            print("Vanilla:", vanilla[0])
            print("SFC    :", sfc[0])
            return [vanilla[0], sfc[0]]

    raise RuntimeError("Could not find solution folders automatically.")

SOLUTION_DIRS = find_solution_dirs()

# Debug
for folder in SOLUTION_DIRS:
    files = list(folder.glob("*.txt"))
    print(folder, "->", len(files), "files")

OUTPUT_DIR = Path("../../tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Parsing helpers
# =========================
def read_solution_file(path: Path) -> dict:
    data = {
        "file": str(path),
        "method": path.parent.name.replace("_subset", ""),
    }

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line == "solution":
                break

            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                continue

            key, value = parts

            if key in {"run", "seed", "iteration_limit", "destroy_percent"}:
                data[key] = int(value)
            elif key in {
                "runtime",
                "objective",
                "total_bin_cost",
                "total_excess",
                "total_conflicts",
            }:
                data[key] = float(value)
            else:
                data[key] = value

    data["instance_name"] = Path(data["instance_file"]).stem
    return data


def parse_set1_instance(name: str):
    # Correia_Random_x_y_z_t
    m = re.match(r"Correia_Random_(\d+)_(\d+)_(\d+)_(\d+)$", name)
    if not m:
        return None

    x, y, z, t = map(int, m.groups())
    return {
        "set": "set1",
        "x": x,
        "y": y,
        "z": z,
        "t": t,
    }


def parse_set2_instance(name: str):
    # HS_Random_x_y_z
    m = re.match(r"HS_Random_(\d+)_(\d+)_(\d+)$", name)
    if not m:
        return None

    x, y, z = map(int, m.groups())
    return {
        "set": "set2",
        "x": x,
        "y": y,
        "z": z,
    }


def add_instance_metadata(row: dict) -> dict:
    name = row["instance_name"]

    meta = parse_set1_instance(name)
    if meta is None:
        meta = parse_set2_instance(name)

    if meta is None:
        row["set"] = "unknown"
        return row

    row.update(meta)
    return row


# =========================
# Read all files
# =========================
rows = []

for folder in SOLUTION_DIRS:
    for path in folder.glob("*.txt"):
        row = read_solution_file(path)
        row = add_instance_metadata(row)
        rows.append(row)

df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError("No solution files found.")

# =========================
# Table 1: Set 1
# For each bin size config
# =========================
set1 = df[df["set"] == "set1"].copy()

table1 = (
    set1
    .groupby(["method", "config", "instance_name"], as_index=False)
    .agg(
        best_obj=("objective", "min"),
        avg_obj=("objective", "mean"),
        mean_runtime=("runtime", "mean"),
        runs=("objective", "count"),
    )
    .sort_values(["config", "instance_name", "method"])
)

table1.to_csv(OUTPUT_DIR / "table1_set1.csv", index=False)

# Optional: split by bin-size config
for config, sub in table1.groupby("config"):
    sub.to_csv(OUTPUT_DIR / f"table1_set1_{config}.csv", index=False)

# =========================
# Table 2: Set 2
# For each cost config
# =========================
set2 = df[df["set"] == "set2"].copy()

table2 = (
    set2
    .groupby(["method", "config", "instance_name"], as_index=False)
    .agg(
        best_obj=("objective", "min"),
        avg_obj=("objective", "mean"),
        mean_runtime=("runtime", "mean"),
        runs=("objective", "count"),
    )
    .sort_values(["config", "instance_name", "method"])
)

table2.to_csv(OUTPUT_DIR / "table2_set2.csv", index=False)

# Optional: split by cost config
for config, sub in table2.groupby("config"):
    sub.to_csv(OUTPUT_DIR / f"table2_set2_{config}.csv", index=False)

# =========================
# Save raw parsed data too
# =========================
df.to_csv(OUTPUT_DIR / "all_runs_raw.csv", index=False)

print("Created:")
print(OUTPUT_DIR / "all_runs_raw.csv")
print(OUTPUT_DIR / "table1_set1.csv")
print(OUTPUT_DIR / "table2_set2.csv")

Found solution folders:
Vanilla: /home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/lnsa_solutions/vanilla_subset
SFC    : /home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/lnsa_solutions/sfc_subset
/home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/lnsa_solutions/vanilla_subset -> 1500 files
/home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/lnsa_solutions/sfc_subset -> 1500 files
Created:
../tables/all_runs_raw.csv
../tables/table1_set1.csv
../tables/table2_set2.csv
